# Ingesta bronze - orders
Se lee `orders.csv` desde la capa raw con un schema definido y se carga en `bronze.orders` con las columnas de auditoria.

In [ ]:
%run ./00_utils

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

params = obtener_parametros()
CATALOGO, STORAGE = params["catalogo"], params["storage"]
ENTIDAD = "orders"
TABLA_DESTINO = f"{CATALOGO}.bronze.{ENTIDAD}"

In [ ]:
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("user_id", IntegerType(), True),
    StructField("eval_set", StringType(), True),
    StructField("order_number", IntegerType(), True),
    StructField("order_dow", IntegerType(), True),
    StructField("order_hour_of_day", IntegerType(), True),
    StructField("days_since_prior_order", DoubleType(), True),
])

In [ ]:
try:
    ruta = ruta_raw(STORAGE, ENTIDAD)
    log(f"Leyendo {ruta}", entidad=ENTIDAD)

    df_raw = (spark.read
              .option("header", True)
              .option("sep", ";")
              .schema(orders_schema)
              .csv(ruta))

    df_bronze = agregar_auditoria(df_raw)
    validar_no_vacio(df_bronze, ENTIDAD)
    escribir_tabla(df_bronze, TABLA_DESTINO, modo="overwrite")

except Exception as e:
    log(f"Error en la ingesta: {e}", "ERROR", ENTIDAD)
    raise

In [ ]:
display(spark.table(TABLA_DESTINO).limit(10))